# Create Health Research Council of New Zealand (HRC) Awards

Creates OpenAlex award rows from HRC's own public Research Repository.

**Data source:** `https://www.hrc.govt.nz/resources/research-repository` (server-rendered listing `?page=0..N` + per-study pages `/resources/research-repository/{slug}`). MBIE's national 'Who got funded' dataset explicitly EXCLUDES HRC, so this first-party repository is authoritative.
**S3 location:** `s3a://openalex-ingest/awards/hrc_nz/hrc_nz_projects.parquet`
**OpenAlex funder:** `F4320334749` - Health Research Council of New Zealand (ROR `https://ror.org/00zbf3d93`, DOI `10.13039/501100001505`), country `NZ`.
**Provenance:** `hrc_research_repository`.
**Priority:** 204.

**Schema choices:**
- One row per HRC-funded study (repository slug).
- Stable `funder_award_id = hrc-{slug}` from the scraper.
- `amount` from the study's `Approved budget:` ($...), `currency = NZD`.
- `funding_type` from `Proposal type:` (Fellowship/Training/Scholarship -> fellowship/training; else research); `funder_scheme = proposal_type`.
- `lead_investigator` from `Researchers:` (leading academic title stripped, name split given/family) with `Host:` as affiliation (country NZ).
- `description` from the study's Lay summary.
- HRC publishes year + duration but not full dates, so `start_year` is populated and `start_date`/`end_date` remain NULL.

## Step 1: Create Raw Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.hrc_nz_raw
USING delta
AS
SELECT
    *,
    current_timestamp() AS databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/hrc_nz/hrc_nz_projects.parquet`;

In [ ]:
%sql
SELECT COUNT(*) AS total_hrc_nz_projects FROM openalex.awards.hrc_nz_raw;

## Step 1.5: Inspect Raw Data, Money Fields, and Key Uniqueness

Expected source columns: `funder_award_id`, `slug`, `display_name`, `description`, `researchers_raw`, `host_organization`, `approved_budget_raw`, `year_raw`, `duration_raw`, `duration_months`, `health_issue`, `proposal_type`, `amount`, `currency`, `start_year`, `landing_page_url`, `downloaded_at`.

In [ ]:
%sql
DESCRIBE openalex.awards.hrc_nz_raw;

In [ ]:
%sql
SELECT funder_award_id, display_name, researchers_raw, host_organization,
       approved_budget_raw, amount, currency, start_year, proposal_type, landing_page_url
FROM openalex.awards.hrc_nz_raw
LIMIT 10;

In [ ]:
%sql
SELECT
    COUNT(*) AS total_rows,
    COUNT(display_name) AS has_title,
    COUNT(description) AS has_description,
    COUNT(researchers_raw) AS has_researchers,
    COUNT(host_organization) AS has_host,
    COUNT(TRY_CAST(amount AS DOUBLE)) AS has_amount,
    COUNT(TRY_CAST(start_year AS INT)) AS has_start_year,
    MIN(TRY_CAST(start_year AS INT)) AS min_start_year,
    MAX(TRY_CAST(start_year AS INT)) AS max_start_year,
    COUNT(DISTINCT funder_award_id) AS distinct_funder_award_ids,
    ROUND(try_divide(COUNT(TRY_CAST(amount AS DOUBLE)), COUNT(*)) * 100.0, 1) AS pct_amount
FROM openalex.awards.hrc_nz_raw;

## Step 1.6: Fail-Fast - Verify HRC Funder Exists

The transform LEFT JOINs to `openalex.common.funder` with a hardcoded fallback (the Abel/MinCiencias canonical-funder gap pattern): if the dim is missing F4320334749 the awards must NOT drop to zero.

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi
FROM openalex.common.funder
WHERE funder_id = 4320334749;

## Step 2: Transform to OpenAlex Awards Schema

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.hrc_nz_awards
USING delta
AS
WITH funder_resolved AS (
    SELECT
        4320334749 AS funder_id,
        COALESCE(MAX(display_name), 'Health Research Council of New Zealand') AS display_name,
        COALESCE(MAX(ror_id), 'https://ror.org/00zbf3d93') AS ror_id,
        COALESCE(MAX(doi), '10.13039/501100001505') AS doi
    FROM openalex.common.funder
    WHERE funder_id = 4320334749
),
raw_prepped AS (
    SELECT
        *,
        TRY_CAST(amount AS DOUBLE) AS amount_double,
        CASE WHEN TRY_CAST(start_year AS INT) BETWEEN 1800 AND 2100 THEN TRY_CAST(start_year AS INT) END AS start_year_int,
        NULLIF(TRIM(host_organization), '') AS leader_affiliation_name,
        TRIM(REGEXP_REPLACE(
            researchers_raw,
            '^((Emeritus|Distinguished|Associate|Adjunct|Clinical|Honorary)\\s+)?(Professor|Prof\\.?|Dr\\.?|Doctor|Mr\\.?|Mrs\\.?|Ms\\.?|Miss|Sir|Dame)\\s+',
            ''
        )) AS leader_clean_name
    FROM openalex.awards.hrc_nz_raw
),
raw_named AS (
    SELECT
        *,
        CASE WHEN leader_clean_name IS NOT NULL AND length(leader_clean_name) > 0
             THEN element_at(split(leader_clean_name, ' '), -1) END AS leader_family_name,
        CASE WHEN size(split(leader_clean_name, ' ')) > 1
             THEN array_join(slice(split(leader_clean_name, ' '), 1, size(split(leader_clean_name, ' ')) - 1), ' ') END AS leader_given_name
    FROM raw_prepped
)
SELECT
    abs(xxhash64(CONCAT('4320334749:', LOWER(TRIM(rp.funder_award_id))))) % 9000000000 AS id,
    rp.display_name,
    NULLIF(TRIM(rp.description), '') AS description,
    4320334749 AS funder_id,
    rp.funder_award_id,
    rp.amount_double AS amount,
    CASE WHEN rp.amount_double IS NOT NULL THEN 'NZD' ELSE NULL END AS currency,
    struct(
        CONCAT('https://openalex.org/F', TRY_CAST(f.funder_id AS STRING)) AS id,
        f.display_name,
        f.ror_id,
        f.doi
    ) AS funder,
    CASE
        WHEN rp.proposal_type ILIKE '%fellowship%' THEN 'fellowship'
        WHEN rp.proposal_type ILIKE '%scholarship%' THEN 'fellowship'
        WHEN rp.proposal_type ILIKE '%career%' THEN 'fellowship'
        WHEN rp.proposal_type ILIKE '%training%' THEN 'training'
        ELSE 'research'
    END AS funding_type,
    NULLIF(TRIM(rp.proposal_type), '') AS funder_scheme,
    'hrc_research_repository' AS provenance,
    CAST(NULL AS DATE) AS start_date,
    CAST(NULL AS DATE) AS end_date,
    rp.start_year_int AS start_year,
    CAST(NULL AS INT) AS end_year,
    CASE
        WHEN rp.leader_family_name IS NOT NULL OR rp.leader_affiliation_name IS NOT NULL THEN
            struct(
                rp.leader_given_name AS given_name,
                rp.leader_family_name AS family_name,
                CAST(NULL AS STRING) AS orcid,
                CAST(NULL AS DATE) AS role_start,
                struct(
                    rp.leader_affiliation_name AS name,
                    CASE WHEN rp.leader_affiliation_name IS NOT NULL THEN 'NZ' ELSE NULL END AS country,
                    CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) AS ids
                ) AS affiliation
            )
        ELSE NULL
    END AS lead_investigator,
    CAST(NULL AS STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >) AS co_lead_investigator,
    CAST(NULL AS ARRAY<STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >>) AS investigators,
    rp.landing_page_url,
    CAST(NULL AS STRING) AS doi,
    CONCAT('https://api.openalex.org/works?filter=awards.id:G',
           TRY_CAST(abs(xxhash64(CONCAT('4320334749:', LOWER(TRIM(rp.funder_award_id))))) % 9000000000 AS STRING)) AS works_api_url,
    current_timestamp() AS created_date,
    current_timestamp() AS updated_date
FROM raw_named rp
CROSS JOIN funder_resolved f
WHERE rp.funder_award_id IS NOT NULL
  AND rp.display_name IS NOT NULL;

## Step 3: Insert into `openalex_awards_raw` at Priority 204

In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'hrc_research_repository' AND priority = 204;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id,
    amount, currency, funder, funding_type, funder_scheme, provenance,
    start_date, end_date, start_year, end_year,
    lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url,
    created_date, updated_date,
    204 AS priority
FROM openalex.awards.hrc_nz_awards;

## Step 6: Verification

In [ ]:
%sql
SELECT COUNT(*) AS total_hrc_nz_awards FROM openalex.awards.hrc_nz_awards;

In [ ]:
%sql
-- Duplicate funder_award_id / id must be zero.
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT funder_award_id) AS distinct_funder_award_ids,
    COUNT(DISTINCT id) AS distinct_openalex_award_ids,
    COUNT(*) - COUNT(DISTINCT funder_award_id) AS duplicate_funder_award_ids,
    COUNT(*) - COUNT(DISTINCT id) AS duplicate_openalex_ids
FROM openalex.awards.hrc_nz_awards;

In [ ]:
%sql
-- Completeness.
SELECT
    COUNT(*) AS total,
    COUNT(display_name) AS has_title,
    COUNT(description) AS has_description,
    SUM(CASE WHEN amount > 0 THEN 1 ELSE 0 END) AS has_positive_amount,
    SUM(CASE WHEN lead_investigator.family_name IS NOT NULL THEN 1 ELSE 0 END) AS has_lead_name,
    SUM(CASE WHEN lead_investigator.affiliation.name IS NOT NULL THEN 1 ELSE 0 END) AS has_lead_affiliation,
    COUNT(start_year) AS has_start_year,
    COUNT(funder_scheme) AS has_funder_scheme,
    ROUND(try_divide(COUNT(display_name), COUNT(*)) * 100.0, 1) AS pct_title,
    ROUND(try_divide(SUM(CASE WHEN amount > 0 THEN 1 ELSE 0 END), COUNT(*)) * 100.0, 1) AS pct_positive_amount,
    ROUND(try_divide(SUM(CASE WHEN lead_investigator.family_name IS NOT NULL THEN 1 ELSE 0 END), COUNT(*)) * 100.0, 1) AS pct_lead_name,
    collect_set(currency) AS currencies
FROM openalex.awards.hrc_nz_awards;

In [ ]:
%sql
-- Amount coverage (not waived: HRC publishes Approved budget).
SELECT
    COUNT(*) AS total,
    COUNT(amount) AS has_amount,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_amount,
    collect_set(currency) AS currencies,
    MIN(amount) AS min_amount, MAX(amount) AS max_amount,
    ROUND(AVG(amount), 2) AS avg_amount, percentile_approx(amount, 0.5) AS median_amount
FROM openalex.awards.hrc_nz_awards;

In [ ]:
%sql
-- PI frequency (Step 6.4a): no name should dominate; no institution-as-name.
SELECT lead_investigator.given_name AS given, lead_investigator.family_name AS family, COUNT(*) AS n
FROM openalex.awards.hrc_nz_awards
GROUP BY 1, 2 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
SELECT funding_type, funder_scheme, COUNT(*) AS awards, ROUND(SUM(amount), 0) AS nzd_total
FROM openalex.awards.hrc_nz_awards
GROUP BY funding_type, funder_scheme ORDER BY awards DESC LIMIT 30;

In [ ]:
%sql
SELECT id, SUBSTRING(display_name, 1, 90) AS title, funder_award_id, amount, currency,
       start_year, lead_investigator.given_name AS given, lead_investigator.family_name AS family,
       SUBSTRING(lead_investigator.affiliation.name, 1, 60) AS affiliation
FROM openalex.awards.hrc_nz_awards
ORDER BY start_year DESC, id LIMIT 20;

In [ ]:
%sql
-- Confirm the priority insert landed.
SELECT priority, provenance, COUNT(*) AS rows
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'hrc_research_repository'
GROUP BY priority, provenance;